# Land to Raw (02b, task `land_to_raw`)

Runs first in the job, before validation. The upload app writes files to the
landing zone (`land_root` = `{env}_banking_and_payments_cat.reference_db.vol_bap_land/dvr`).
For every landed file (valid or not), this task:

1. Copies it to the legacy NetApp R: drive drop location with the OEDIS DataBridge
   SMB handler (`oedis.handler.smb.upload`). DataBridge reads from the landing zone,
   so its account (`lake-{env}-oedisint`) only needs access to `vol_bap_land`.
2. Moves it from the landing zone to the raw inbox (`raw_root/inbox`), where
   validation and the rest of the pipeline pick it up.

The R: drive copy keeps the existing DataStage process loading FMBODW during
the migration transition period. If the copy fails, the files are still moved
to raw so they are processed, and the task fails so notify alerts support.

**Requires classic compute** — serverless cannot reach on-prem NetApp.

Credentials:
- Service account: `bocad\triton_s`
- Key Vault scope: configured via `secret_scope` variable
- UNC path: parameterised per environment in `databricks.yml`

In [0]:
%run ../../common

In [0]:
import json
import logging
import os
import shutil
from datetime import datetime

# Job parameters. DataBridge connection settings come from the shared job
# context (_ctx) built by common.py, which resolves them from Key Vault.
dbutils.widgets.text("legacy_r_drive_path", "", "Legacy R drive UNC path")

logger = logging.getLogger(__name__)
r_drive_path = dbutils.widgets.get("legacy_r_drive_path").strip()

# The app uploads to the landing zone. Copy every landed file as uploaded;
# validation (on the raw inbox) does not filter the R: drive copy.
land_root = _ctx["land_root"].rstrip("/")
inbox_root = f'{_ctx["raw_root"].rstrip("/")}/inbox'
try:
    file_names = sorted(
        name for name in os.listdir(land_root)
        if os.path.isfile(os.path.join(land_root, name))
    )
except FileNotFoundError:
    file_names = []
logger.info("Found %d file(s) in %s", len(file_names), land_root)


def _move_to_raw(names):
    """Move landed files to the raw inbox (copy, then delete from landing)."""
    os.makedirs(inbox_root, exist_ok=True)
    for name in names:
        shutil.copyfile(os.path.join(land_root, name), os.path.join(inbox_root, name))
        os.remove(os.path.join(land_root, name))
    logger.info("Moved %d file(s) from %s to %s", len(names), land_root, inbox_root)
    return list(names)


def _exit(status, copied=0, **extra):
    # Lets notify show the copy outcome; a missing value means the copy failed.
    dbutils.jobs.taskValues.set(key="status", value=status)
    dbutils.notebook.exit(json.dumps({
        "copied_count": copied,
        "r_drive_path": r_drive_path,
        "status": status,
        "copied_at": datetime.utcnow().isoformat(),
        **extra,
    }))


if not file_names:
    _exit("NO_FILES")
if not r_drive_path:
    moved = _move_to_raw(file_names)
    _exit("SKIPPED_NO_R_DRIVE_PATH", files=file_names, moved_to_raw=len(moved))

# Imported only when there is work to do.
from oeds.domain.utils.ingest.ingest_utils import DisConnectionOptions, run_ingest

scope = _ctx["oeds_secrete_scope"]
connection = DisConnectionOptions(
    host=_ctx["internal_databricks_workspace_url"],
    azure_tenant_id=_ctx["azure_tenant_id"],
    azure_client_id=dbutils.secrets.get(scope=scope, key=_ctx["service_spn_id_key"]),
    azure_client_secret=dbutils.secrets.get(scope=scope, key=_ctx["service_spn_secrete_key"]),
    azure_workspace_resource_id=_ctx["internal_databricks_resource_id"],
    dis_job_id=int(_ctx["internal_dis_job_id"]),
)

logger.info("Copying %d file(s) from %s to %s", len(file_names), land_root, r_drive_path)
print(f"Copying {file_names} from {land_root} to {r_drive_path}")

copy_error = None
try:
    result = run_ingest(
        connection=connection,
        config_file="default.json",
        template_params={
            "r_drive_path": r_drive_path,
            "source_root": land_root,
            "file_names": file_names,
        },
        # SMB destinations can't be listed from Databricks, so skip file verification.
        # run_ingest waits for DataBridge to finish, so the files are only moved
        # out of the landing zone after the copy is done.
        verification=None,
    )
except Exception as exc:
    copy_error = exc
    logger.error("DataBridge R drive transfer failed: %s", exc)

# Move to raw even if the R: drive copy failed, so the files are still processed.
moved = _move_to_raw(file_names)

if copy_error is not None:
    # Fail the task (no status value) so notify reports the R: drive copy failure.
    raise copy_error

ingest = result.ingest_result
logger.info("DataBridge R drive transfer completed")
_exit(
    "SUBMITTED",
    copied=len(file_names),
    files=file_names,
    moved_to_raw=len(moved),
    databridge_status=str(ingest.get_status()),
    databridge_run_url=str(ingest.get_job_run_page_url()),
)